# Module 1: What Predicts GDP?

Our dataset contains **493 features** for **254 countries**, each labelled as one of three roles:

- **Causal** -- features with a plausible economic mechanism linking them to GDP  
- **Spurious** -- features that *should not* matter (Scrabble scores, flag colors, numerology...)  
- **Incidental** -- real-world statistics that correlate with GDP but are not root causes  

In this module you will browse the features, plot them against GDP, and discover that **spurious features can correlate just as strongly as causal ones**.

In [ ]:
# ── Colab Setup (auto-skipped when running locally) ──────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    GITHUB_RAW_BASE = (
        "https://raw.githubusercontent.com/eth-bmai-fs26/coding-exercises"
        "/week1/cx1_un-games"
    )

    import subprocess, urllib.request

    # 1. Install packages not pre-installed in Colab
    subprocess.run(["pip", "install", "-q", "ipywidgets"], check=True)

    # 2. Download data files into /content (Colab's working directory)
    for fname in ["gdp_spurious_regression_dataset.csv", "codebook.csv"]:
        if not os.path.exists(fname):
            print(f"Downloading {fname}...")
            urllib.request.urlretrieve(f"{GITHUB_RAW_BASE}/{fname}", fname)

    # 3. Download styling.py and __init__.py from GitHub
    helpers_dir = "/content/helpers"
    os.makedirs(helpers_dir, exist_ok=True)
    for hfile in ["__init__.py", "styling.py"]:
        dest = os.path.join(helpers_dir, hfile)
        if not os.path.exists(dest):
            urllib.request.urlretrieve(
                f"{GITHUB_RAW_BASE}/notebooks/helpers/{hfile}", dest
            )

    # 4. Write data_loader.py directly (ensures CWD-first path resolution)
    with open(os.path.join(helpers_dir, "data_loader.py"), "w") as f:
        f.write('''"""Shared data loading for GDP Spurious Regression notebooks."""
import os, pandas as pd

def _find_file(filename):
    cwd_path = os.path.join(os.getcwd(), filename)
    if os.path.exists(cwd_path):
        return cwd_path
    base_dir = os.path.join(os.path.dirname(__file__), '..', '..')
    return os.path.join(base_dir, filename)

def load_dataset():
    df = pd.read_csv(_find_file('gdp_spurious_regression_dataset.csv'), index_col=0)
    codebook = pd.read_csv(_find_file('codebook.csv'))
    y = df['gdp_per_capita_usd']
    X = df.drop(columns=['gdp_per_capita_usd'])
    role_map = dict(zip(codebook['column_name'], codebook['role']))
    return X, y, codebook, role_map
''')

    # 5. Ensure /content is on sys.path so `import helpers` works
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')

print("Setup complete.")

In [ ]:
#@title 🔧 Setup { display-mode: "form" }

import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from IPython.display import display, clear_output
import ipywidgets as widgets
from ipywidgets import Button, VBox, HBox, Output, HTML, Dropdown, ToggleButtons, Checkbox, Layout

from helpers import load_dataset, ROLE_COLORS, apply_dark_theme, takeaway_box, metric_card, role_bar_colors
from helpers.styling import role_legend_html, styled_button, DARK_BG, AXES_BG, TEXT_COLOR, GREEN, RED, GRAY, MUTED_TEXT

# Load data
X, y, codebook, role_map = load_dataset()
N_FEATURES = X.shape[1]

# Pre-compute correlations for efficiency
correlations = X.select_dtypes(include='number').corrwith(y).dropna()


# ── Widget 1: Feature Browser ──────────────────────────────────────────────

class FeatureBrowser:
    """Browse features grouped by role with descriptions from the codebook."""

    def __init__(self):
        self.role_toggle = ToggleButtons(
            options=['causal', 'spurious', 'incidental'],
            description='Role:',
            style={'button_width': '130px', 'description_width': '50px'},
        )
        self.role_toggle.observe(self._on_role_change, names='value')
        self.output = Output()
        self._render_table('causal')

    def _render_table(self, role):
        subset = codebook[codebook['role'] == role][['column_name', 'description']].reset_index(drop=True)
        color = ROLE_COLORS.get(role, GRAY)
        rows = ''.join(
            f"<tr style='border-bottom:1px solid #2a2a4a;'>"
            f"<td style='padding:6px 12px; color:{MUTED_TEXT}; font-family:monospace; font-size:13px;'>{r.column_name}</td>"
            f"<td style='padding:6px 12px; color:{TEXT_COLOR}; font-size:13px;'>{r.description}</td>"
            f"</tr>"
            for _, r in subset.iterrows()
        )
        html = (
            f"<div style='max-height:350px; overflow-y:auto; border-radius:10px; border:1px solid #475569;'>"
            f"<table style='width:100%; border-collapse:collapse; background:{AXES_BG};'>"
            f"<thead><tr style='background:{color}22;'>"
            f"<th style='padding:8px 12px; text-align:left; color:{color}; font-size:13px;'>Feature</th>"
            f"<th style='padding:8px 12px; text-align:left; color:{color}; font-size:13px;'>Description</th>"
            f"</tr></thead><tbody>{rows}</tbody></table></div>"
            f"<p style='color:{MUTED_TEXT}; font-size:12px; margin-top:6px;'>{len(subset)} features</p>"
        )
        with self.output:
            clear_output(wait=True)
            display(HTML(html))

    def _on_role_change(self, change):
        self._render_table(change['new'])

    def create_ui(self):
        title = HTML(f"<h3 style='color:{TEXT_COLOR}; margin-bottom:4px;'>Feature Browser</h3>")
        return VBox([title, self.role_toggle, self.output],
                    layout=Layout(padding='15px', margin='10px 0'))


# ── Widget 2: Scatter Plot Explorer ────────────────────────────────────────

class ScatterExplorer:
    """Scatter plot of any numeric feature vs GDP per capita."""

    def __init__(self):
        numeric_cols = X.select_dtypes(include='number').columns.tolist()
        options = sorted(
            [(f"[{role_map.get(c, '?').capitalize()}] {c}", c) for c in numeric_cols],
            key=lambda t: t[0]
        )
        self.dropdown = Dropdown(
            options=options,
            description='Feature:',
            style={'description_width': '70px'},
            layout=Layout(width='500px'),
        )
        self.log_cb = Checkbox(value=False, description='Log-scale GDP (y-axis)',
                               style={'description_width': 'initial'})
        self.dropdown.observe(self._update, names='value')
        self.log_cb.observe(self._update, names='value')
        self.output = Output()
        self._update(None)

    def _update(self, _):
        col = self.dropdown.value
        role = role_map.get(col, 'incidental')
        color = ROLE_COLORS.get(role, GRAY)
        mask = X[col].notna() & y.notna()
        xv, yv = X.loc[mask, col], y[mask]

        with self.output:
            clear_output(wait=True)
            if len(xv) < 3:
                display(HTML(f"<p style='color:{RED};'>Not enough data for {col}</p>"))
                return

            fig, ax = plt.subplots(figsize=(8, 4.5))
            apply_dark_theme(fig, ax)

            y_plot = np.log10(yv) if self.log_cb.value else yv
            ax.scatter(xv, y_plot, s=22, alpha=0.65, color=color, edgecolors='none')

            r_val, p_val = pearsonr(xv, yv)
            ax.set_xlabel(col.replace('_', ' ').title())
            ylabel = 'log10(GDP per capita USD)' if self.log_cb.value else 'GDP per capita (USD)'
            ax.set_ylabel(ylabel)
            ax.set_title(f'{col}  vs  GDP per capita   (R = {r_val:.3f},  p = {p_val:.2e})',
                         fontsize=12)

            plt.tight_layout()
            plt.show()

    def create_ui(self):
        title = HTML(f"<h3 style='color:{TEXT_COLOR}; margin-bottom:4px;'>Scatter Plot Explorer</h3>")
        controls = HBox([self.dropdown, self.log_cb])
        return VBox([title, controls, self.output],
                    layout=Layout(padding='15px', margin='10px 0'))


# ── Widget 3: All Correlations Bar Chart ───────────────────────────────────

class TopCorrelations:
    """Horizontal bar chart of ALL features by |correlation| with GDP."""

    def __init__(self):
        self.output = Output()
        self._render()

    def _render(self):
        # Sort all features by |correlation|
        sorted_corr = correlations.reindex(correlations.abs().sort_values(ascending=True).index)
        features = sorted_corr.index.tolist()
        values = sorted_corr.values
        colors = role_bar_colors(features, role_map)

        n = len(features)
        fig_height = max(8, n * 0.06)

        with self.output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, fig_height))
            apply_dark_theme(fig, ax)

            ax.barh(range(n), values, color=colors, edgecolor='none', height=0.8)
            ax.set_yticks(range(0, n, max(1, n // 40)))
            ax.set_yticklabels([features[i].replace('_', ' ') for i in range(0, n, max(1, n // 40))], fontsize=7)
            ax.set_xlabel('Pearson r with GDP per capita')
            ax.set_title(f'All {n} Features by Correlation with GDP', fontsize=13)

            plt.tight_layout()
            plt.show()

    def create_ui(self):
        title = HTML(f"<h3 style='color:{TEXT_COLOR}; margin-bottom:4px;'>All Feature Correlations with GDP</h3>")
        legend = role_legend_html()
        return VBox([title, legend, self.output],
                    layout=Layout(padding='15px', margin='10px 0'))


# ── Widget 4: Guess the Category Quiz ──────────────────────────────────────

QUIZ_FEATURES = [
    'life_expectancy_at_birth',
    'scrabble_score_country_name',
    'internet_users_pct',
    'flag_colors_count',
    'tertiary_enrollment_gross',
    'name_numerology_score',
    'rule_of_law_index',
]


class GuessTheCategory:
    """Interactive quiz: guess whether a feature is causal, spurious, or incidental."""

    def __init__(self):
        self.idx = 0
        self.features = [f for f in QUIZ_FEATURES if f in X.columns]

        self.guess_toggle = ToggleButtons(
            options=['Causal', 'Spurious', 'Incidental'],
            description='Your guess:',
            style={'button_width': '120px', 'description_width': '80px'},
        )
        self.reveal_btn = styled_button('Reveal Answer', width='160px')
        self.next_btn = styled_button('Next Feature', color='#475569', width='160px')
        self.reveal_btn.on_click(self._on_reveal)
        self.next_btn.on_click(self._on_next)

        self.plot_output = Output()
        self.info_output = Output()
        self.feedback_output = Output()

        self._show_current()

    def _current_feature(self):
        return self.features[self.idx % len(self.features)]

    def _show_current(self):
        col = self._current_feature()
        desc_row = codebook[codebook['column_name'] == col]
        desc = desc_row['description'].values[0] if len(desc_row) > 0 else ''
        mask = X[col].notna() & y.notna()
        xv, yv = X.loc[mask, col], y[mask]

        with self.info_output:
            clear_output(wait=True)
            display(HTML(
                f"<div style='background:{AXES_BG}; padding:12px 18px; border-radius:10px; "
                f"border:1px solid #475569; margin-bottom:8px;'>"
                f"<p style='color:{TEXT_COLOR}; font-size:15px; margin:0 0 4px 0;'>"
                f"<b>Feature {self.idx + 1}/{len(self.features)}:</b> "
                f"<code style='color:#a5b4fc;'>{col}</code></p>"
                f"<p style='color:{MUTED_TEXT}; font-size:13px; margin:0;'>{desc}</p>"
                f"</div>"
            ))

        with self.plot_output:
            clear_output(wait=True)
            if len(xv) < 3:
                display(HTML(f"<p style='color:{RED};'>Not enough data.</p>"))
                return
            fig, ax = plt.subplots(figsize=(7, 4))
            apply_dark_theme(fig, ax)
            ax.scatter(xv, yv, s=20, alpha=0.6, color='#60a5fa', edgecolors='none')
            r_val, _ = pearsonr(xv, yv)
            ax.set_xlabel(col.replace('_', ' ').title())
            ax.set_ylabel('GDP per capita (USD)')
            ax.set_title(f'R = {r_val:.3f}', fontsize=12)
            plt.tight_layout()
            plt.show()

        with self.feedback_output:
            clear_output(wait=True)

    def _on_reveal(self, _):
        col = self._current_feature()
        true_role = role_map.get(col, 'incidental')
        guess = self.guess_toggle.value.lower()
        correct = (guess == true_role)
        icon = '&#10003;' if correct else '&#10007;'
        bg = '#22c55e33' if correct else '#ef444433'
        border_color = GREEN if correct else RED
        role_color = ROLE_COLORS.get(true_role, GRAY)

        with self.feedback_output:
            clear_output(wait=True)
            display(HTML(
                f"<div style='background:{bg}; padding:12px 18px; border-radius:10px; "
                f"border-left:4px solid {border_color}; margin-top:8px;'>"
                f"<span style='color:{border_color}; font-size:18px; font-weight:bold;'>{icon}</span> "
                f"<span style='color:{TEXT_COLOR}; font-size:14px;'>{'Correct!' if correct else 'Not quite.'} "
                f"This feature is </span>"
                f"<span style='color:{role_color}; font-weight:bold; font-size:14px;'>{true_role.upper()}</span>"
                f"</div>"
            ))

    def _on_next(self, _):
        self.idx += 1
        if self.idx >= len(self.features):
            self.idx = 0
        self._show_current()

    def create_ui(self):
        title = HTML(f"<h3 style='color:{TEXT_COLOR}; margin-bottom:4px;'>Guess the Category</h3>")
        subtitle = HTML(
            f"<p style='color:{MUTED_TEXT}; font-size:13px; margin-bottom:8px;'>"
            f"Look at the scatter plot, then guess whether the feature is Causal, Spurious, or Incidental.</p>"
        )
        buttons = HBox([self.reveal_btn, self.next_btn])
        return VBox([title, subtitle, self.info_output, self.plot_output,
                     self.guess_toggle, buttons, self.feedback_output],
                    layout=Layout(padding='15px', margin='10px 0'))


print(f'Setup complete. {N_FEATURES} features loaded.')

## Browse the dataset

Use the toggle below to explore what kinds of features live in each category.  
Pay attention to the *spurious* ones -- some of them will surprise you later.

In [ ]:
#@title ▶️ Launch Feature Browser { display-mode: "form" }
browser = FeatureBrowser()
display(browser.create_ui())

## Scatter plots: feature vs GDP

Pick any feature from the dropdown and see how it relates to GDP per capita.  
The colour of the dots reflects the feature's role. Try toggling the log-scale checkbox for skewed distributions.

In [ ]:
#@title ▶️ Launch Scatter Explorer { display-mode: "form" }
scatter = ScatterExplorer()
display(scatter.create_ui())

## Which features correlate most with GDP?

The bar chart below shows **all features** ranked by the strength of their linear correlation with GDP, color-coded by role.  
Notice how the colours mix -- **spurious features can rank just as high as causal ones**.

In [ ]:
#@title ▶️ Launch Top Correlations { display-mode: "form" }
top_corr = TopCorrelations()
display(top_corr.create_ui())

## Quiz: can you tell real from ridiculous?

For each feature below, look at its scatter plot against GDP and try to guess its category.  
Click **Reveal Answer** to check, then **Next Feature** to continue.

In [ ]:
#@title ▶️ Launch Quiz { display-mode: "form" }
quiz = GuessTheCategory()
display(quiz.create_ui())

## Takeaway

In [ ]:
#@title ▶️ Takeaway { display-mode: "form" }
display(takeaway_box(
    '\U0001f4a1 <b>High correlation \u2260 causation.</b> Starbucks locations correlate with GDP, '
    'but Starbucks doesn\u2019t cause wealth \u2014 wealth causes Starbucks. '
    'In the next module, we\u2019ll see what happens when we try to build a predictive model '
    'with <em>all</em> these features.'
))